# Week 18, Day 2: Permutation tests: testing by shuffling

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w018/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A two-sample permutation test

Write `perm_test(a, b, n_perm, rng)`: pool the data, shuffle, split into groups of the original sizes, and count shuffles whose absolute difference in means is at least the observed one. Apply it to two groups of 15 Gaussian observations and compare with the z-test p-value.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt
def Phi(z): return 0.5 * (1 + erf(z / sqrt(2)))

def perm_test(a, b, n_perm, rng):
    obs = abs(b.mean() - a.mean())
    pooled = np.concatenate([a, b])
    count = 0
    for _ in range(n_perm):
        s = rng.permutation(pooled)
        if ...:
            count += 1
    return ...

r1 = np.random.default_rng(2002)
a, b = r1.normal(0, 1, 15), r1.normal(0.8, 1, 15)
p_perm = perm_test(a, b, 9999, r1)
z = (b.mean() - a.mean()) / np.sqrt(a.var(ddof=1) / 15 + b.var(ddof=1) / 15)
p_z = 2 * (1 - Phi(abs(z)))
print(f"permutation p = {p_perm:.4f}, z-test p = {p_z:.4f}")

In [ ]:
import numpy as np
def test_range():
    "perm_test returns a p-value in (0, 1]"
    p = perm_test(np.array([1.0, 2, 3]), np.array([4.0, 5, 6]), 999, np.random.default_rng(0))
    assert 0 < p <= 1

def test_identical():
    "identical groups give a large p-value"
    assert perm_test(np.ones(10), np.ones(10), 999, np.random.default_rng(0)) == 1.0

def test_agrees():
    "for Gaussian data, the permutation p-value is within 0.02 of the z-test p-value"
    assert abs(p_perm - p_z) < 0.02

test_range()
test_identical()
test_agrees()
print("All checks passed")

### Exercise: The sign-flip test

Write `signflip_test(d, n_perm, rng)` using random signs (±1) for each difference, and apply it to yesterday's drifting neuron: 40 trials with baseline and response counts.

In [ ]:
def signflip_test(d, n_perm, rng):
    obs = abs(d.mean())
    signs = rng.choice([-1, 1], size=(n_perm, len(d)))
    null = ...                                # absolute mean of each sign-flipped set
    return ...

r2 = np.random.default_rng(2)
drift = 10 + 8 * np.sin(np.linspace(0, 3 * np.pi, 40))
d = r2.poisson(drift + 1.5) - r2.poisson(drift)
p_flip = signflip_test(d, 9999, r2)
print(f"sign-flip p = {p_flip:.4f}")

In [ ]:
import numpy as np
def test_function():
    "signflip_test returns 1 for differences that are all zero"
    assert signflip_test(np.zeros(8), 999, np.random.default_rng(0)) == 1.0

def test_detects():
    "it detects the response in the drifting neuron (p below 0.05)"
    assert p_flip < 0.05

test_function()
test_detects()
print("All checks passed")

### Exercise: Correct across neurons with the maximum statistic

Write `max_stat_test(D, n_perm, rng)` for a matrix D of paired differences (trials × neurons): flip the signs of whole trials (the same flips for all neurons), record the maximum over neurons of |mean / standard error| on each flip, and return the neurons whose observed statistic exceeds the 95th percentile. Then estimate the FWER over 200 null experiments, with and without correction, and apply it to a dataset where neurons 0 to 4 respond.

In [ ]:
def tstats(D):
    return np.abs(D.mean(axis=0) / (D.std(axis=0, ddof=1) / np.sqrt(len(D))))

def max_stat_test(D, n_perm, rng):
    obs = tstats(D)
    maxes = np.empty(n_perm)
    for i in range(n_perm):
        flips = rng.choice([-1, 1], size=(len(D), 1))      # one sign per trial, shared by all neurons
        maxes[i] = ...
    thresh = np.percentile(maxes, 95)
    return np.flatnonzero(obs > thresh)

r3 = np.random.default_rng(50)
n_tr, n_neur = 30, 50
any_fp_raw, any_fp_max = [], []
for _ in range(200):
    D0 = r3.normal(size=(n_tr, n_neur))                   # no neuron responds
    any_fp_raw.append(np.any(tstats(D0) > 1.96))
    any_fp_max.append(len(max_stat_test(D0, 300, r3)) > 0)
fwer_uncorrected, fwer_max = np.mean(any_fp_raw), np.mean(any_fp_max)

D1 = r3.normal(size=(n_tr, n_neur)); D1[:, :5] += 1.0         # neurons 0 to 4 respond
found = max_stat_test(D1, 2000, r3)
print(f"family-wise error: uncorrected {fwer_uncorrected:.2f}, maximum statistic {fwer_max:.2f};  neurons found: {found}")

In [ ]:
import numpy as np
def test_fwer():
    "over 200 null experiments, uncorrected testing gives a false positive in most, the maximum statistic in about 5% (at most 10%)"
    assert fwer_uncorrected > 0.6 and fwer_max <= 0.1

def test_finds_real():
    "in the dataset with 5 responsive neurons, the max-statistic method finds at least 3 of them and no others"
    assert set(found) <= set(range(5)) and len(found) >= 3

test_fwer()
test_finds_real()
print("All checks passed")